import dependencies

In [129]:
import random
from qdrant_client import QdrantClient

import instructor
import json
from pydantic import BaseModel, Field

from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
import tiktoken

In [130]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [131]:
qdrant_client = QdrantClient(url="http://localhost:6333")

load all the data from qdrant

In [132]:
all_points = []
next_offset = None
batch_size = 100

while True:
    points, next_offset = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        limit=batch_size,
        offset=next_offset,
        with_payload=True,
        with_vectors=False
    )
    all_points.extend(points)
    if next_offset is None:
        break

In [133]:
len(all_points)

1000

split the data into 2 groups : 50 items for synthetic question generation , 950 items to loop through and evaluate their relevance to previous generated synthetic question

In [134]:
rng = random.Random(42)

In [135]:
indices = list(range(len(all_points)))

In [136]:
indices

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [137]:
rng.shuffle(indices)

In [138]:
rng.shuffle(indices)

In [139]:
sampl_idx = set(indices[:50])

In [140]:
sampl_idx

{35,
 85,
 114,
 129,
 134,
 136,
 143,
 152,
 160,
 173,
 180,
 198,
 210,
 217,
 229,
 265,
 273,
 274,
 280,
 285,
 323,
 333,
 344,
 371,
 388,
 441,
 456,
 508,
 538,
 579,
 586,
 661,
 663,
 667,
 690,
 710,
 750,
 755,
 776,
 781,
 782,
 842,
 851,
 860,
 876,
 882,
 897,
 931,
 975,
 996}

In [141]:
sample_50 = [p for i, p in enumerate(all_points) if i in sampl_idx]

In [142]:
sample_50

[Record(id=36, payload={'preprocessed_description': 'UtechSmart USB Hub, 4-Port USB 3.0 Hub, Ultra-Slim USB Adapter for Laptop, 5Gbps Fast Data Transfer, USB 3.0 Splitter for Dell, HP, MacBook, Surface Pro, Flash Drive, HDD, Keyboard, Mouse, PS4 【Up to SuperSpeed】UetchSmart 4-port USB 3.0 hub provides a 5 Gbps data transfer speed which is more than 10 times faster than USB 2.0. Sync data at Stable and blazing speeds let you move files quickly and efficiently. 【Borderless to Use】This USB splitter for laptop is compatible for Windows, and macOS systems. It works with PC, MacBook, iMac, PS4, Switches, Flash Drives, Keyboard, Mouse & any other USB compatible devices, backward support with USB 1.0, USB 1.1 and USB 2.0 devices. 【Plug and Play】UtechSmart USB expander hub with a built-in high-performance chip supports OTG and hot swapping. Plug in up to 4 USB devices using a single USB 3.0 port. Note: the hub is not designed for charging. (Max output is only 900mA.) 【Premium and Precision】Utec

In [143]:
remaining_points = [p for i, p in enumerate(all_points) if i not in sampl_idx]

In [144]:
len(sample_50)

50

In [145]:
len(remaining_points)

950

Generate synthetic questions

In [146]:
all_context_sample_50 = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in sample_50]

In [147]:
all_context_sample_50

[{'id': 'B0B5ZY1RPL',
  'text': 'UtechSmart USB Hub, 4-Port USB 3.0 Hub, Ultra-Slim USB Adapter for Laptop, 5Gbps Fast Data Transfer, USB 3.0 Splitter for Dell, HP, MacBook, Surface Pro, Flash Drive, HDD, Keyboard, Mouse, PS4 【Up to SuperSpeed】UetchSmart 4-port USB 3.0 hub provides a 5 Gbps data transfer speed which is more than 10 times faster than USB 2.0. Sync data at Stable and blazing speeds let you move files quickly and efficiently. 【Borderless to Use】This USB splitter for laptop is compatible for Windows, and macOS systems. It works with PC, MacBook, iMac, PS4, Switches, Flash Drives, Keyboard, Mouse & any other USB compatible devices, backward support with USB 1.0, USB 1.1 and USB 2.0 devices. 【Plug and Play】UtechSmart USB expander hub with a built-in high-performance chip supports OTG and hot swapping. Plug in up to 4 USB devices using a single USB 3.0 port. Note: the hub is not designed for charging. (Max output is only 900mA.) 【Premium and Precision】UtechSmart USB Data hub 

In [148]:
all_context_remaining_points = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in remaining_points]

In [149]:
len(all_context_remaining_points)

950

In [150]:
SYSTEM_PROMPT = f"""
You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
- Refer to the items as "products". Never use the word "chunk".
- Keep questions specific. Even in the full 1000-product collection, a good question should be answerable from at most 4 products. Avoid broad or generic questions that a large number of products could satisfy.

## Products
{json.dumps(all_context_sample_50, indent=2)}
"""

In [151]:
print(SYSTEM_PROMPT)


You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
-

In [152]:
client = instructor.from_provider(
    "openai/gpt-5.4",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [153]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

In [154]:
class SyntheticQuestion(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question: str = Field(description="Suggested question.")
    chunk_ids: list[str] = Field(description="ID of the chunk that could be used to answer the question.")
    answer_example: str = Field(description="Suggested answer grounded in the context.")

class SyntheticQuestions(BaseModel):
    synthetic_questions: list[SyntheticQuestion] = Field(description="List of synthetic questions, 30 in total")

In [155]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestions
)

In [156]:
len(response.synthetic_questions)

30

In [157]:
response.synthetic_questions

[SyntheticQuestion(reasoning='This asks about a product with four USB 3.0 ports and a 5Gbps transfer speed, which is uniquely described by the UtechSmart hub.', question="I'm looking for a compact USB hub with 4 USB 3.0 ports and 5Gbps transfer speed for my laptop. Which product fits that?", chunk_ids=['B0B5ZY1RPL'], answer_example='The UtechSmart USB Hub (B0B5ZY1RPL) fits that. It has 4 USB 3.0 ports, supports up to 5Gbps data transfer, and is designed as a compact ultra-slim hub for laptops.'),
 SyntheticQuestion(reasoning='The question targets true wireless earbuds with ANC and a specific battery life, all found in one product listing.', question='Do you have true wireless earbuds with active noise cancellation and around 32 hours of total battery life?', chunk_ids=['B09WZDKSF7'], answer_example='Yes. The Skullcandy Indy ANC Fuel True Wireless Earbuds (B09WZDKSF7) offer active noise cancellation and up to 32 hours total battery life.'),
 SyntheticQuestion(reasoning='The product desc

In [158]:
for s_question in response.synthetic_questions:
    print(s_question.chunk_ids)

['B0B5ZY1RPL']
['B09WZDKSF7']
['B0BTYQRQBN']
['B0BWV5L9JP']
['B0B53LT5K6']
['B09RMJWBSM']
['B00XU1MXCY']
['B09XTJ97BR']
['B0BVZMKTRL']
['B0BJ1J36PJ']
['B0BB6TFQ22']
['B0BKVX86MD']
['B07KTN8SQW']
['B09YXPJ16J']
['B0C4GDFRDD']
['B0BJ1J36PJ', 'B0BT917GSS']
['B09YTG56Y6', 'B0BKSNPSQF']
['B09YHG2Z7F', 'B0BYMMB4LY']
['B0B53LT5K6', 'B09QCDP12Q']
['B09WZDKSF7', 'B0CG5L93XZ']
['B0BN33YT98', 'B0C61XW574']
['B0BB6TFQ22', 'B0BV6YQHHH']
['B0B5ZY1RPL', 'B0BJ1J36PJ']
['B09RMJWBSM', 'B0BTLTFCVR']
['B0BG4B4HKF', 'B0BBFWC6TF']
[]
[]
[]
[]
[]


In [159]:
synthetic_questions_relevant_data = [{"question_id": i, "question": item.question} for i, item in enumerate(response.synthetic_questions)]

In [160]:
synthetic_questions_relevant_data

[{'question_id': 0,
  'question': "I'm looking for a compact USB hub with 4 USB 3.0 ports and 5Gbps transfer speed for my laptop. Which product fits that?"},
 {'question_id': 1,
  'question': 'Do you have true wireless earbuds with active noise cancellation and around 32 hours of total battery life?'},
 {'question_id': 2,
  'question': 'Which product is a USB-C to USB-C fast charging cable that shows the charging power on the cable itself?'},
 {'question_id': 3,
  'question': 'I want a slim wireless keyboard that can switch between my laptop and phone and has a built-in cradle. Which product should I get?'},
 {'question_id': 4,
  'question': "Do you have a smartwatch that's IP68 waterproof and tracks heart rate, blood oxygen, and sleep?"},
 {'question_id': 5,
  'question': 'Which product lets me convert an SFP port to RJ45 Ethernet for up to 100 meters?'},
 {'question_id': 6,
  'question': 'I need a flat Cat7 Ethernet cable with 10Gbps speed for running along a wall. Which product matc

Assign remaining chunks to relevant questions

In [161]:
SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
{json.dumps(all_context_remaining_points[0:50], indent=2)}
"""

In [162]:
print(SYSTEM_PROMPT_950)


You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
[
  {
    "question_id": 0,
    "question": "I'm looking for a compact USB hub with 4 USB 3.0 ports and 5Gbps transfer speed for my laptop. Which product fits that

In [163]:
def count_tokens(text):

    encoding = tiktoken.get_encoding("o200k_base")

    return len(encoding.encode(text))

Count number of tokens for static prefix

In [164]:
SYSTEM_PROMPT_STATIC_PART = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products"""

In [165]:
def count_tokens(text):

    encoding = tiktoken.get_encoding("o200k_base")

    return len(encoding.encode(text))

In [166]:
print(count_tokens(SYSTEM_PROMPT_STATIC_PART))

1349


Assign chunks to relevant questions for a single batch

In [167]:
client = instructor.from_provider(
    "openai/gpt-5.4-mini",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [168]:
class SyntheticQuestionMatchingChunk(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question_id: int = Field(description="ID of the question.")
    chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")

class SyntheticQuestionMatchingChunks(BaseModel):
    synthetic_question_matching_chunks: list[SyntheticQuestionMatchingChunk] = Field(description="List of synthetic questions and their matching chunks")

In [169]:
tmp_response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT_950}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestionMatchingChunks
)

In [170]:
tmp_response.synthetic_question_matching_chunks

[SyntheticQuestionMatchingChunk(reasoning='The question asks for a compact USB hub with 4 USB 3.0 ports and 5Gbps transfer speed. Product B0BKPB2YQ9 includes 3 USB 3.0 ports at up to 5 Gbps, but not 4 ports, so it does not fully satisfy the question. No other product matches all required specs independently.', question_id=0, chunk_ids=[]),
 SyntheticQuestionMatchingChunk(reasoning='The question asks for true wireless earbuds with active noise cancellation and around 32 hours total battery life. Product B0BSDT6YYJ is true wireless earbuds with ANC and about 30+ hours total playtime, which is close to 32 hours and sufficient to answer the question.', question_id=1, chunk_ids=['B0BSDT6YYJ']),
 SyntheticQuestionMatchingChunk(reasoning='The question asks for a USB-C to USB-C fast charging cable that shows charging power on the cable itself. Product B0CDRNNRCK is a USB-C to USB-C 100W cable with a smart LED display that shows real-time charging power.', question_id=2, chunk_ids=['B0CDRNNRCK'

In [171]:
for data in tmp_response.synthetic_question_matching_chunks:
    print(f"Question ID: {data.question_id}, Chunk IDs: {data.chunk_ids}")

Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk IDs: ['B0BSDT6YYJ']
Question ID: 2, Chunk IDs: ['B0CDRNNRCK']
Question ID: 3, Chunk IDs: []
Question ID: 4, Chunk IDs: []
Question ID: 5, Chunk IDs: []
Question ID: 6, Chunk IDs: []
Question ID: 7, Chunk IDs: []
Question ID: 8, Chunk IDs: []
Question ID: 9, Chunk IDs: []
Question ID: 10, Chunk IDs: []
Question ID: 11, Chunk IDs: []
Question ID: 12, Chunk IDs: []
Question ID: 13, Chunk IDs: []
Question ID: 14, Chunk IDs: []
Question ID: 15, Chunk IDs: []
Question ID: 16, Chunk IDs: ['B09X1LDMH6']
Question ID: 17, Chunk IDs: []
Question ID: 18, Chunk IDs: ['B0CFLR6R3X', 'B09Q5TNDHY']
Question ID: 19, Chunk IDs: ['B0BSDT6YYJ']
Question ID: 20, Chunk IDs: []
Question ID: 21, Chunk IDs: []
Question ID: 22, Chunk IDs: []
Question ID: 23, Chunk IDs: []
Question ID: 24, Chunk IDs: ['B0BLTPRG3X']
Question ID: 25, Chunk IDs: []
Question ID: 26, Chunk IDs: []
Question ID: 27, Chunk IDs: []
Question ID: 28, Chunk IDs: []
Question ID: 29, Chunk ID

In [172]:
raw_response.usage

ResponseUsage(input_tokens=19075, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=1801, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=20876)

In [173]:
def get_relevant_chunks(synthetic_questions, remaining_points_batch):

    SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions, indent=2)}

## Products
{json.dumps(remaining_points_batch, indent=2)}
"""

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_950}
        ],
        reasoning={"effort": "none"},
        response_model=SyntheticQuestionMatchingChunks
    )

    return response.synthetic_question_matching_chunks

In [174]:
answer_extended_chunks = []

for i in range(0, len(all_context_remaining_points), 50):

    print(f"Working on batch {(i+50)/50}")

    answer_extended_chunks.append(get_relevant_chunks(synthetic_questions_relevant_data, all_context_remaining_points[i:i+50]))

Working on batch 1.0
Working on batch 2.0
Working on batch 3.0
Working on batch 4.0
Working on batch 5.0
Working on batch 6.0
Working on batch 7.0
Working on batch 8.0
Working on batch 9.0
Working on batch 10.0
Working on batch 11.0
Working on batch 12.0
Working on batch 13.0
Working on batch 14.0
Working on batch 15.0
Working on batch 16.0
Working on batch 17.0
Working on batch 18.0
Working on batch 19.0


In [175]:
len(answer_extended_chunks)

19

In [176]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        print(f"Question ID: {item.question_id}, Chunk IDs: {item.chunk_ids}")

Batch 1
Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk IDs: ['B09YWSG8NR']
Question ID: 2, Chunk IDs: ['B0CDRNNRCK']
Question ID: 3, Chunk IDs: []
Question ID: 4, Chunk IDs: []
Question ID: 5, Chunk IDs: []
Question ID: 6, Chunk IDs: []
Question ID: 7, Chunk IDs: []
Question ID: 8, Chunk IDs: []
Question ID: 9, Chunk IDs: []
Question ID: 10, Chunk IDs: []
Question ID: 11, Chunk IDs: []
Question ID: 12, Chunk IDs: []
Question ID: 13, Chunk IDs: []
Question ID: 14, Chunk IDs: []
Question ID: 15, Chunk IDs: []
Question ID: 16, Chunk IDs: []
Question ID: 17, Chunk IDs: []
Question ID: 18, Chunk IDs: []
Question ID: 19, Chunk IDs: []
Question ID: 20, Chunk IDs: []
Question ID: 21, Chunk IDs: []
Question ID: 22, Chunk IDs: []
Question ID: 23, Chunk IDs: []
Question ID: 24, Chunk IDs: []
Question ID: 25, Chunk IDs: []
Question ID: 26, Chunk IDs: []
Question ID: 27, Chunk IDs: []
Question ID: 28, Chunk IDs: []
Question ID: 29, Chunk IDs: []
Batch 2
Question ID: 0, Chunk IDs: []
Batch 3
Qu

In [177]:
questions_container = {i: [] for i in range(30)}

In [178]:
questions_container

{0: [],
 1: [],
 2: [],
 3: [],
 4: [],
 5: [],
 6: [],
 7: [],
 8: [],
 9: [],
 10: [],
 11: [],
 12: [],
 13: [],
 14: [],
 15: [],
 16: [],
 17: [],
 18: [],
 19: [],
 20: [],
 21: [],
 22: [],
 23: [],
 24: [],
 25: [],
 26: [],
 27: [],
 28: [],
 29: []}

In [179]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        questions_container[item.question_id].extend(item.chunk_ids)

Batch 1
Batch 2
Batch 3
Batch 4
Batch 5
Batch 6
Batch 7
Batch 8
Batch 9
Batch 10
Batch 11
Batch 12
Batch 13
Batch 14
Batch 15
Batch 16
Batch 17
Batch 18
Batch 19


In [180]:
for i, data in enumerate(response.synthetic_questions):
    questions_container[i].extend(data.chunk_ids)

In [181]:
questions_container

{0: ['B0BFWZ1JPY', 'B0B46D4FM3', 'B0B5ZY1RPL'],
 1: ['B09YWSG8NR',
  'B0BK8WW5DH',
  'B0C7HY8NKW',
  'B0C5H9XMS8',
  'B0BQJ5H3H2',
  'B0BRH3JTWJ',
  'B09WZDKSF7'],
 2: ['B0CDRNNRCK', 'B09V7VC1LB', 'B0B1HQV2GT', 'B0BTYQRQBN'],
 3: ['B09QPQZMMN', 'B0BWV5L9JP'],
 4: ['B09MZ9BZP5',
  'B0BYRXT52V',
  'B09KLYPYTN',
  'B0C7CTNS12',
  'B0C6F2PSDX',
  'B09V2Y1TQB',
  'B0BKZ7QG61',
  'B09YHJYM9V',
  'B0B2ZVX1GQ',
  'B0B53LT5K6'],
 5: ['B09RMJWBSM'],
 6: ['B00XU1MXCY'],
 7: ['B0BC8YQ14K', 'B09XTJ97BR'],
 8: ['B0BVZMKTRL'],
 9: ['B0BYRX9BF2', 'B0BVDKD6JD', 'B0CF575R2D', 'B0B28KBV3H', 'B0BJ1J36PJ'],
 10: ['B09WVKFC9Z', 'B0BB6TFQ22'],
 11: ['B0BCF4D9PP', 'B0BKVX86MD'],
 12: ['B07KTN8SQW'],
 13: ['B09V481PDD',
  'B0BTKX543T',
  'B0C9712HYL',
  'B09XFFW3XC',
  'B0BXC72RLD',
  'B09YXPJ16J'],
 14: ['B0C4GDFRDD'],
 15: ['B0B5KVF7YL', 'B0BVH2C84P', 'B0B2571M6J', 'B0BJ1J36PJ', 'B0BT917GSS'],
 16: ['B0B2J5JHC8',
  'B0B8ZMQ53J',
  'B0CH8F91CP',
  'B0BJZ9DXBB',
  'B0C8HS6143',
  'B09YTG56Y6',
  'B0BKSNPSQF'],

Adjust the golden truth answers for the original synthetic questions by adding new grounding context

In [182]:
def get_description(parent_asin: str) -> str:
    
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["preprocessed_description"]

In [183]:
get_description("B09KLYPYTN")

"Smart Watch Compatible with iPhone Android Phones 2022, 1.7 inch Watches for Men Women IP68 Waterproof Fitness Tracker with Blood Oxygen/Heart Rate Monitor Sleep Steps Tracker DIY Watch Faces Black 【Activity & Fitness Trackers】- The fitness tracker watch features a 1.7-inch full-touch high-definition color screen, giving you a different visual experience. Supports up to 20 sports modes, it can accurately record your daily steps, calories, distance, speed and other data, help you easily grasp the daily sports data. 【Waterproof Smart Watch with Personalized Dial 】- There are up to 100 cloud watch dials to choose from, and the cloud watch dials will be continuously updated. You can also use any picture you like to create a personalized watch dial. The IP68 waterproof design allows you to have no worries when washing your hands, running in the rain, swimming in the pool or other water activities. (It can't track your swimming data) 【Heart Rate Monitor & Blood Oxygen Monitor】- Built-in opt

In [184]:
def generate_golden_answer(question, relevant_chunks):

    class GoldenAnswer(BaseModel):
        answer: str = Field(description="The ideal reference answer for the question.")

    PROMPT = f"""
You are the shopping assistant of an e-shop. You are also being used to generate the ideal reference answer for a single customer question, to serve as ground truth for evaluating a RAG system.

## Context
A customer asked a question. A retrieval system has already gathered the products most relevant to it. Your job is to write the best possible answer a helpful shopping assistant could give, using ONLY those products.

## Inputs
- `question`: the customer's question.
- `products`: a list of relevant products, each with a `text` description. This may be empty or may not actually contain the answer.

## How to write the answer
- Ground every factual claim strictly in the provided product descriptions. Never invent products, prices, specs, availability, or details that are not present in the text.
- Answer in the voice of a friendly, concise shopping assistant speaking directly to the customer.
- Use only the information needed to answer well; don't dump every product if only some are relevant.
- If the products only partially answer the question, answer what you can and clearly state what information isn't available.
- If the products do not contain the information needed to answer at all (or the list is empty), do not guess. Respond the way a good assistant would when the shop doesn't carry the item or the info isn't available (e.g. politely say you couldn't find it), and do not fabricate an answer.

## Question
{json.dumps(question, indent=2)}

## Products
{json.dumps(relevant_chunks, indent=2)}"""

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": PROMPT}
        ],
        reasoning={"effort": "none"},
        response_model=GoldenAnswer
    )

    return response.answer

In [185]:
reference_dataset = []

for i, data in enumerate(response.synthetic_questions):

    print(f"Working on question {i+1}")

    relevant_chunks = [get_description(item) for item in questions_container[i]]

    golden_answer = generate_golden_answer(data.question, relevant_chunks)

    reference_dataset.append({
        "question": data.question,
        "ground_truth": golden_answer,
        "reference_context_ids": questions_container[i],
        "reference_descriptions": relevant_chunks
    })

Working on question 1
Working on question 2
Working on question 3
Working on question 4
Working on question 5
Working on question 6
Working on question 7
Working on question 8
Working on question 9
Working on question 10
Working on question 11
Working on question 12
Working on question 13
Working on question 14
Working on question 15
Working on question 16
Working on question 17
Working on question 18
Working on question 19
Working on question 20
Working on question 21
Working on question 22
Working on question 23
Working on question 24
Working on question 25
Working on question 26
Working on question 27
Working on question 28
Working on question 29
Working on question 30


In [186]:
reference_dataset

[{'question': "I'm looking for a compact USB hub with 4 USB 3.0 ports and 5Gbps transfer speed for my laptop. Which product fits that?",
  'ground_truth': 'The best match is the UtechSmart USB Hub. It has 4 USB 3.0 ports, supports 5Gbps data transfer speed, and is described as ultra-slim and compact, so it fits your laptop use case well. The LYFNLOVE and barsone hubs also have 4 USB 3.0 ports and up to 5Gbps speed, but the UtechSmart is specifically described as ultra-slim and portable.',
  'reference_context_ids': ['B0BFWZ1JPY', 'B0B46D4FM3', 'B0B5ZY1RPL'],
  'reference_descriptions': ['4-Port USB Hub 3.0 with USB C Adapter，LYFNLOVE USB Extender with Individual Power Switch, Ultra Slim USB Splitter for Laptop, MacBook Pro/Air/Mini, iMac, PC, Surface Pro, PS4, PS5, Xbox 4 Port USB 3.0 Hub: LYFNLOVE USB splitter extend 1 USB port to 4, let you connect multiple USB devices at same time, such as flash drive, keyboard, mouse, card reader, printer, camera, USB fan and USB cables. USB3.0 Sup

Create LangSmith Dataset

In [187]:
ls_client = Client()

In [188]:
dataset_name = "rag-evaluation-dataset-extended"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset for 1000 items"
)

In [189]:
for item in reference_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"],
        },
        outputs={
            "ground_truth": item["ground_truth"],
            "reference_context_ids": item["reference_context_ids"],
            "reference_descriptions": item["reference_descriptions"]
        }
    )